# FIFA World Cup 2026 Predictor
This notebook analyzes historical football data and uses Groq AI to predict match outcomes.

In [ ]:
import os
import json
from zipfile import ZipFile
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

# Enhance display
pd.set_option('display.max_columns', None)

## Setup Environment Variables

In [ ]:
load_dotenv()
# Ensure GEMINI_API_KEY is loaded
if not os.getenv('GEMINI_API_KEY'):
    print('Please set your GEMINI_API_KEY in the .env file')

## Load Dataset

In [ ]:
DATASET = './archive.zip'

def load_results(path=DATASET):
    if path.endswith('.zip'):
        with ZipFile(path) as z:
            return pd.read_csv(z.open('results.csv'), parse_dates=['date'])
    return pd.read_csv(path, parse_dates=['date'])

df = load_results()
df.head()

## Feature Engineering & Statistics

In [ ]:
def form(team, n=5):
    matches = df[(df.home_team == team) | (df.away_team == team)].sort_values('date').tail(n)
    result = []
    for _, row in matches.iterrows():
        if row.home_team == team:
            scored, conceded = row.home_score, row.away_score
        else:
            scored, conceded = row.away_score, row.home_score
        if scored > conceded: result.append('W')
        elif scored == conceded: result.append('D')
        else: result.append('L')
    return '-'.join(result) if result else 'N/A'

def stats(team):
    matches = df[(df.home_team == team) | (df.away_team == team)].sort_values('date')
    if len(matches) == 0: return None
    wins = draws = losses = gf = ga = 0
    for _, row in matches.iterrows():
        if row.home_team == team:
            scored, conceded = row.home_score, row.away_score
        else:
            scored, conceded = row.away_score, row.home_score
        gf += scored
        ga += conceded
        if scored > conceded: wins += 1
        elif scored == conceded: draws += 1
        else: losses += 1
    return {
        'matches': len(matches),
        'wins': wins,
        'draws': draws,
        'losses': losses,
        'goals_for': gf,
        'goals_against': ga,
        'avg_goals': round(gf / len(matches), 2),
        'avg_conceded': round(ga / len(matches), 2),
        'recent_form': form(team),
    }

In [ ]:
def h2h(team_a, team_b):
    matches = df[
        ((df.home_team == team_a) & (df.away_team == team_b)) |
        ((df.home_team == team_b) & (df.away_team == team_a))
    ]
    a = b = d = 0
    for _, row in matches.iterrows():
        if row.home_score == row.away_score:
            d += 1
        elif (row.home_team == team_a and row.home_score > row.away_score) or (row.away_team == team_a and row.away_score > row.home_score):
            a += 1
        else:
            b += 1
    return {'team_a_wins': a, 'draws': d, 'team_b_wins': b}

## Prediction Function using Gemini

In [ ]:
def predict(team_a, team_b, stage='Group Stage', model='gemini-3.1-flash-lite'):
    team_a_stats = stats(team_a)
    team_b_stats = stats(team_b)
    if team_a_stats is None or team_b_stats is None:
        raise ValueError('Unknown team.')
    head2head = h2h(team_a, team_b)
    llm = ChatGoogleGenerativeAI(model=model, google_api_key=os.getenv('GEMINI_API_KEY'), temperature=0.2)
    prompt = f"""
You are a professional football analyst.
Predict the FIFA World Cup 2026 match.
Stage:\n{stage}\nTeam A:\n{team_a}\nStatistics:\n{json.dumps(team_a_stats, indent=2)}
Team B:\n{team_b}\nStatistics:\n{json.dumps(team_b_stats, indent=2)}
Head-to-head:\n{json.dumps(head2head, indent=2)}
Use: Historical performance, Recent form, Goals scored, Goals conceded, Head-to-head, Overall team quality
Return ONLY valid JSON.
{{
    \"predicted_winner\":\"\",
    \"confidence_percentage\":0,
    \"key_reasoning\":\"\"
}}
"""
    response = llm.invoke([('system', 'You are an expert football analyst. Return ONLY JSON.'), ('human', prompt)])
    content = response.content.strip()
    if content.startswith('```'):
        content = content.replace('```json', '').replace('```', '').strip()
    return json.loads(content)

## Run Prediction

In [ ]:
result = predict('Paraguay', 'France')
print('\n==============================')
print(' FIFA WORLD Cup 2026 PREDICTION')
print('==============================')
print(f"Winner      : {result['predicted_winner']}")
print(f"Confidence  : {result['confidence_percentage']}%")
print(f"Reason      : {result['key_reasoning']}")

## Data Visualization (Enhancement)

In [ ]:
team = 'France'
matches = df[(df.home_team == team) | (df.away_team == team)].copy()
matches['year'] = matches['date'].dt.year
goals_per_year = matches.groupby('year').apply(lambda x: pd.Series({
    'Goals For': sum(x.home_score[x.home_team == team]) + sum(x.away_score[x.away_team == team]),
    'Goals Against': sum(x.away_score[x.home_team == team]) + sum(x.home_score[x.away_team == team])
})).reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=goals_per_year, x='year', y='Goals For', label='Goals For')
sns.lineplot(data=goals_per_year, x='year', y='Goals Against', label='Goals Against')
plt.title(f'{team} - Goals Scored vs Conceded Over Time')
plt.xlabel('Year')
plt.ylabel('Goals')
plt.legend()
plt.show()